# Exp 35: Training curves of submissions

Card: [exp35_training_curves.md](../../experiments/exp35_training_curves.md)

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from core.config import ROOT
from core.cv import CVResult, Config
from core.tracking import TrackedRun

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    FOLDS = [0]
else:
    FOLDS = [0, 1, 2, 3, 4]
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
MODELS = {
    "exp11": Config.from_toml(CONFIG_DIR / "exp11_resnet18_5fold" / "resnet18.toml"),
    "exp26": Config.from_toml(CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.toml"),
    "exp29": Config.from_toml(CONFIG_DIR / "exp29_downsample_5fold" / "resnet18.toml"),
}
FIGURES = ROOT / "sections" / "model" / "figures"
PREFIX = "exp35_training_curves"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode
SERIES = {"train": "#eb6834", "validation": "#2a78d6"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)


## Per-epoch history

In [ ]:
METRICS = {"train_loss_epoch": ("loss", "train"), "val_loss": ("loss", "validation"),
           "train_map": ("mAP", "train"), "val_map": ("mAP", "validation")}


def history(config, fold):
    """Per-epoch metrics of fold run of `fold`, long format; epoch = rank of logged step."""
    run_id = CVResult.load(config).run_ids[fold]
    rows = []
    for key, (metric, split) in METRICS.items():
        rows += [{"epoch": i + 1, "metric": metric, "split": split, "value": v} for i, v in enumerate(TrackedRun(run_id).curve(key))]
    return pd.DataFrame(rows).assign(fold=fold)


curves = pd.concat({name: pd.concat([history(c, f) for f in FOLDS]) for name, c in MODELS.items()}, names=["model"])
curves = curves.reset_index(level=0)

## Loss and mAP curves

In [ ]:
figures = {}
for metric in ["loss", "mAP"]:
    fig, axes = plt.subplots(1, len(MODELS), figsize=(4.2 * len(MODELS), 3.6), sharey=True)
    for name, ax in zip(MODELS, axes):
        part = curves[(curves["model"] == name) & (curves["metric"] == metric)]
        for (split, fold), line in part.groupby(["split", "fold"]):
            ax.plot(line["epoch"], line["value"], color=SERIES[split], lw=1, alpha=0.7, label=split if fold == FOLDS[0] else None)
        ax.set(title=name, xlabel="epoch", xticks=range(1, int(part["epoch"].max()) + 1))
    axes[0].set_ylabel(metric)
    axes[0].legend(loc="best")
    fig.suptitle(f"{metric} per epoch, one line per fold", color="#52514e")
    fig.tight_layout()
    figures[f"{PREFIX}_{metric.lower()}.png"] = fig
plt.show()

In [ ]:
if SAVE_FIGURES:
    for fname, fig in figures.items():
        fig.savefig(FIGURES / fname, dpi=120)